# Africa–Europe UN General Debate: NLP & Machine Learning

Professional reconstruction of the **AI-course foundation project**. It exposes the main analytical steps with portable paths and reusable functions.

**Questions:** How similar are African and European diplomatic statements? Which themes and clusters characterize the corpus? How much regional information can a text classifier recover?

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import load_ungd, add_text_features, filter_africa_europe
from src.comparative_analysis import corpus_cosine_similarity, fit_kmeans_tfidf, build_lstm_classifier
from src.sentiment import add_sentiment_features
from src.visualization import make_wordcloud, plot_sentiment_over_time, plot_cluster_distribution


## 1. Load the public UNGD corpus

In [ ]:
df = load_ungd(ROOT / 'data' / 'un-general-debates.csv')
df = df.loc[df['year'].between(1970, 2015)].copy()
df.shape


## 2. Construct Africa and Europe samples

The professional code embeds the Africa/Europe country-code choices from the original coursework mapping, so no separate mapping spreadsheet is required.

In [ ]:
regional = filter_africa_europe(df)
regional = add_text_features(regional)
regional['Continent'].value_counts()


## 3. Regional word clouds

In [ ]:
for continent in ['Africa', 'Europe']:
    part = regional.loc[regional['Continent'].eq(continent), 'processed_text']
    fig, _ = make_wordcloud(part, title=f'{continent}: high-frequency diplomatic vocabulary')


## 4. TF–IDF cosine similarity

The submitted AI project reported **0.2640**. The cell below recomputes the same high-level regional-corpus comparison from the local data snapshot.

In [ ]:
africa = regional.loc[regional['Continent'].eq('Africa'), 'processed_text'].tolist()
europe = regional.loc[regional['Continent'].eq('Europe'), 'processed_text'].tolist()
similarity = corpus_cosine_similarity(africa, europe)
similarity.similarity


## 5. LDA topic comparison

The submitted model used **10 topics**, `no_above=0.30`, `no_below=10`, **50 passes**, and `random_state=0`. Use `src.comparative_analysis.fit_joint_lda` on a common tokenized corpus, then aggregate document-topic probabilities by continent. The original topic interpretations are documented in `docs/findings.md`.

## 6. K-means clustering

In [ ]:
km = fit_kmeans_tfidf(regional['processed_text'].tolist(), n_clusters=3, random_state=42)
regional['cluster'] = km.labels
plot_cluster_distribution(km.labels)
pd.crosstab(regional['Continent'], regional['cluster'])


## 7. LSTM continent classification

The original submission used an 80/20 split, 10,000-word vocabulary, sequence length 100, embedding dimension 100, LSTM(128), Adam learning rate 0.001, batch size 20, and 10 epochs. Its reported held-out accuracy was **85.1%**. TensorFlow is optional; install `requirements-ai.txt` before reproducing this section.

In [ ]:
# Architecture only; tokenization/splitting should be performed explicitly before fitting.
model = build_lstm_classifier()
model.summary()


## 8. VADER sentiment: Africa vs Europe

The professional rebuild scores sentence-like units from the **original speech text**. The submitted sentiment figures remain documented as coursework outputs.

In [ ]:
sentiment = add_sentiment_features(regional, text_column='text')
plot_sentiment_over_time(sentiment, group_column='Continent', title='Africa vs Europe: mean VADER sentiment')


## Interpretation boundary

Similarity, topics, clusters, sentiment and classification accuracy are descriptive measurements derived from text. They do **not** identify causal effects of continent or historical events on diplomatic rhetoric.